# 09 · Métricas, calibración y selección de umbral

Un modelo puede tener buen ranking y malas probabilidades, o alta accuracy y ser inútil para la clase que importa. Este lab separa **discriminación, calibración y decisión**.

## Objetivos
- Derivar precision, recall, specificity y F1 desde la matriz de confusión.
- Entender ROC-AUC vs PR-AUC.
- Evaluar probabilidades con log-loss y Brier score.
- Construir reliability diagrams.
- Calibrar con sigmoid/Platt e isotonic.
- Elegir thresholds según costo, capacidad operacional o restricciones.


In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import *
from sklearn.calibration import calibration_curve, CalibratedClassifierCV, CalibrationDisplay
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
SEED=42
X,y=make_classification(n_samples=6000,n_features=20,n_informative=8,n_redundant=4,weights=[.94,.06],flip_y=.01,random_state=SEED)
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.3,stratify=y,random_state=SEED)

## 1. Matriz de confusión
Para clase positiva:
- TP: detectado correctamente.
- FP: alarma falsa.
- FN: caso positivo perdido.
- TN: negativo correcto.

$Precision=TP/(TP+FP)$, $Recall=TP/(TP+FN)$, $Specificity=TN/(TN+FP)$.

El costo real depende del dominio: en screening puede priorizarse recall; en revisión manual limitada puede priorizarse precision.


In [ ]:
model=RandomForestClassifier(n_estimators=500,min_samples_leaf=3,class_weight='balanced_subsample',random_state=SEED,n_jobs=-1).fit(Xtr,ytr)
p=model.predict_proba(Xte)[:,1]
for th in [.1,.2,.3,.5,.7]:
 pred=(p>=th).astype(int); tn,fp,fn,tp=confusion_matrix(yte,pred).ravel()
 print(th,{'precision':round(precision_score(yte,pred),3),'recall':round(recall_score(yte,pred),3),'specificity':round(tn/(tn+fp),3),'f1':round(f1_score(yte,pred),3),'alerts':int(pred.sum())})

## 2. ROC vs Precision-Recall
ROC grafica TPR contra FPR. Con una clase muy rara, una FPR pequeña puede representar muchas falsas alarmas. PR muestra directamente precision vs recall y suele ser más intuitiva cuando el positivo es escaso.

AUC mide ranking, **no** el threshold operativo. Dos modelos con AUC similar pueden producir comportamientos muy distintos en el punto que interesa.


In [ ]:
print('ROC-AUC',roc_auc_score(yte,p),'PR-AUC',average_precision_score(yte,p))
fig,ax=plt.subplots(1,2,figsize=(11,4)); RocCurveDisplay.from_predictions(yte,p,ax=ax[0]); PrecisionRecallDisplay.from_predictions(yte,p,ax=ax[1]); plt.show()

## 3. Calibración
Si un modelo asigna 0.8 a 100 casos, idealmente cerca de 80 deberían ser positivos. Eso es calibración.

- **Log-loss:** penaliza probabilidades muy confiadas y equivocadas.
- **Brier:** error cuadrático de probabilidades.
- **Reliability diagram:** probabilidad predicha vs frecuencia observada.

Random Forest y boosting pueden rankear muy bien y aun así necesitar calibración.


In [ ]:
print('Brier',brier_score_loss(yte,p),'LogLoss',log_loss(yte,p))
CalibrationDisplay.from_predictions(yte,p,n_bins=10,strategy='quantile'); plt.plot([0,1],[0,1],'--'); plt.title('Antes de calibrar'); plt.show()

## 4. Platt/sigmoid vs isotonic
`sigmoid` aprende una transformación logística y funciona bien con pocos datos. `isotonic` es no paramétrica y flexible, pero puede overfit con muestras pequeñas. **La calibración debe usar datos separados o CV**, nunca el test final.


In [ ]:
base=RandomForestClassifier(n_estimators=350,min_samples_leaf=3,class_weight='balanced_subsample',random_state=SEED,n_jobs=-1)
cal_sig=CalibratedClassifierCV(base,method='sigmoid',cv=5).fit(Xtr,ytr)
cal_iso=CalibratedClassifierCV(base,method='isotonic',cv=5).fit(Xtr,ytr)
fig,ax=plt.subplots(figsize=(6,5))
for name,m in [('raw',model),('sigmoid',cal_sig),('isotonic',cal_iso)]:
 pr=m.predict_proba(Xte)[:,1]; frac,mean=calibration_curve(yte,pr,n_bins=10,strategy='quantile'); ax.plot(mean,frac,'o-',label=f'{name} Brier={brier_score_loss(yte,pr):.3f}')
ax.plot([0,1],[0,1],'--'); ax.legend(); ax.set(xlabel='predicho',ylabel='observado'); plt.show()

## 5. Threshold por costo
Supón que un FN cuesta 10 y un FP cuesta 1. Podemos buscar el threshold que minimiza costo esperado. En otros contextos, la restricción puede ser `recall >= 0.9` o revisar solo 500 casos/día.


In [ ]:
thresholds=np.linspace(.01,.99,99); rows=[]
for th in thresholds:
 pred=(p>=th).astype(int); tn,fp,fn,tp=confusion_matrix(yte,pred).ravel(); cost=10*fn+1*fp
 rows.append([th,cost,precision_score(yte,pred,zero_division=0),recall_score(yte,pred),pred.sum()])
res=pd.DataFrame(rows,columns=['threshold','cost','precision','recall','alerts']); display(res.loc[res.cost.idxmin()])
plt.plot(res.threshold,res.cost); plt.axvline(res.loc[res.cost.idxmin(),'threshold'],ls='--'); plt.ylabel('costo'); plt.xlabel('threshold'); plt.show()

## 6. Precision@K / Recall@K
Cuando solo puedes revisar K casos, a veces ni siquiera necesitas un threshold fijo: ordenas por score y tomas top-K. Esto es común en fraude, auditoría y priorización.


In [ ]:
for k in [50,100,200]:
 idx=np.argsort(-p)[:k]; print('K',k,'precision@k',yte[idx].mean(),'recall@k',yte[idx].sum()/yte.sum())

## Errores comunes
- optimizar accuracy en 1:100 imbalance;
- usar ROC-AUC como única métrica;
- escoger threshold sobre test;
- asumir que 0.8 significa 80% sin revisar calibración;
- reportar F1 sin indicar la clase positiva;
- comparar PR-AUC entre datasets con prevalencias muy distintas sin contexto.

## Ejercicios
1. Optimiza threshold para `recall>=0.90` maximizando precision.
2. Compara Logistic Regression y RF en ranking y calibración.
3. Simula caída de prevalencia y observa precision.
4. Calcula expected calibration error (ECE).
5. Construye curvas de lift y gains.
6. Diseña una política con tres zonas: auto-aprobar, revisión humana, auto-rechazar.
